# https://adventofcode.com/2022

In [1]:
import copy
from functools import cache
from pathlib import Path

import networkx as nx

In [2]:
puzz = """
Valve AA has flow rate=0; tunnels lead to valves DD, II, BB
Valve BB has flow rate=13; tunnels lead to valves CC, AA
Valve CC has flow rate=2; tunnels lead to valves DD, BB
Valve DD has flow rate=20; tunnels lead to valves CC, AA, EE
Valve EE has flow rate=3; tunnels lead to valves FF, DD
Valve FF has flow rate=0; tunnels lead to valves EE, GG
Valve GG has flow rate=0; tunnels lead to valves FF, HH
Valve HH has flow rate=22; tunnel leads to valve GG
Valve II has flow rate=0; tunnels lead to valves AA, JJ
Valve JJ has flow rate=21; tunnel leads to valve II
""".strip().split(
    "\n"
)
puzz = Path("./day16.txt").read_text().splitlines()

In [3]:
puzz[:5]

['Valve XB has flow rate=0; tunnels lead to valves WZ, LE',
 'Valve BM has flow rate=0; tunnels lead to valves PL, RI',
 'Valve GC has flow rate=0; tunnels lead to valves HN, IT',
 'Valve RM has flow rate=0; tunnels lead to valves ZQ, YL',
 'Valve ZM has flow rate=5; tunnels lead to valves SN, KE, UW, MY, GW']

## Part 1

In [4]:
Rog = dict()
Gog = nx.Graph()

for p in puzz:
    words = p.split()
    start, rate, ends = words[1], words[4], words[9:]

    rate = int(rate[5:-1])
    ends = [e.strip(",") for e in ends]
    start_v = f"{start}_v"

    Rog[start] = rate

    # if rate:
    #     G.add_edge(start, start_v, rate=rate)
    for end in ends:
        Gog.add_edge(start, end)
        # Gog.add_edge(end, start)
        # if rate:
        #     G.add_edge(start_v, end)

In [5]:
G = Gog.copy()
all_pairs_shortest = dict(nx.all_pairs_shortest_path_length(G))
rates = Rog.copy()

# nx.draw_networkx(G)

In [6]:
def dfs(tuns, release, cupo, opened):
    cands = []
    for nepo, rate in rates.items():
        if rate <= 0:
            continue
        elif nepo in opened:
            continue
        elif tuns - all_pairs_shortest[cupo][nepo] - 1 < 0:
            continue
        else:
            cands.append(nepo)

    if not cands:
        releases.append(release)
    else:
        for nepo in cands:
            nepo_dst = all_pairs_shortest[cupo][nepo]
            nepo_tun = tuns - nepo_dst - 1
            nepo_rel = release + nepo_tun * rates[nepo]
            nepo_cur = nepo
            nepo_ope = opened.union([nepo])

            dfs(nepo_tun, nepo_rel, nepo_cur, nepo_ope)


releases = []
dfs(
    tuns=30,
    release=0,
    cupo="AA",
    opened=set(),
)

max(releases), releases.__len__(), sorted(releases)[:-5:-1]

(2265, 133405, [2265, 2263, 2257, 2257])

## Part 2

## Stuffs from Reddit

In [ ]:
raise

In [ ]:
import collections as c
import functools
import itertools
import re

r = r"Valve (\w+) .*=(\d*); .* valves? (.*)"

Valves, Flows, Distances = set(), dict(), c.defaultdict(lambda: 1000)

for v, f, us in re.findall(r, "\n".join(puzz)):
    Valves.add(v)  # store node
    if f != "0":
        Flows[v] = int(f)  # store flow
    for u in us.split(", "):
        Distances[u, v] = 1  # store dist

for k, i, j in itertools.product(Valves, Valves, Valves):  # floyd-warshall
    Distances[i, j] = min(Distances[i, j], Distances[i, k] + Distances[k, j])

In [ ]:
@functools.cache
def search(
    t,
    u="AA",
    vs=frozenset(Flows),
    e=False,
    depth=0,
):
    print(" " * (depth * 4) + f"{t:02}", u, f"e={int(e)}", sorted(vs))
    humang = [
        Flows[v] * (t - Distances[u, v] - 1)
        + search(
            t=t - Distances[u, v] - 1,
            u=v,
            vs=vs - {v},
            e=e,
            depth=depth + 1,
        )
        for v in vs
        if Distances[u, v] < t
    ]

    if e:
        elephang = search(
            t=26,
            u="AA",
            vs=vs,
            e=False,
            depth=depth + 1,
        )
    else:
        elephang = 0
    return max(humang + [elephang])


# print(search(30))
print(search(26, e=True))

In [ ]:
@functools.cache
def search1(
    t,
    u="AA",
    vs=frozenset(Flows),
):
    res = [0]
    for v in vs:
        if Distances[u, v] < t:
            cur_pressure = Flows[v] * (t - Distances[u, v] - 1)
            nex_pressure = search1(
                t - Distances[u, v] - 1,
                v,
                vs - {v},
            )
            res.append(cur_pressure + nex_pressure)
    return max(res)


search1(30)

In [ ]:
@functools.cache
def search2(
    t,
    u="AA",
    vs=frozenset(Flows),
):
    res = [0]
    for v in vs:
        if Distances[u, v] < t:
            cur_pressure = Flows[v] * (t - Distances[u, v] - 1)
            nex_pressure = search2(
                t - Distances[u, v] - 1,
                v,
                vs - {v},
            )
            res.append(cur_pressure + nex_pressure)
    res += [search1(26, "AA", vs=vs)]
    return max(res)


search2(26)